<h1>Chapter 11 - Coding Agents</h1>
<i>Create an Agent for developing code.</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 11 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma3:12b &

# ▂▂▂▂▂▂▂▂▂▂▂▂

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [ ]:
import os
from illustrated_agents.llm import LLM

# Ollama
llm = LLM(model="ollama/gemma3:12b")

# Llama.cpp server
# llm = LLM(model="openai/gemma-3-12b-it-Q4_K_M", api_base="http://localhost:8080", api_key="sk-no-key-required")
 
# Llama-cpp-python server
# llm = LLM(model="openai/gemma-3-12b-it-Q4_K_M.gguf", api_base="http://localhost:8000/v1/", api_key="sk-no-key-required")

# LM Studio
# llm = LLM(model="lm_studio/gemma-3-12b-it", api_base="http://localhost:1234/v1", api_key="sk-no-key-required")

# Google's Gemini / Gemma
# os.environ['GEMINI_API_KEY'] = "YOUR_GEMINI_API_KEY"
# llm = LLM(model="gemini/gemini-2.5-flash")
# llm = LLM(model="gemini/gemma-3-12b-it")

## 2 - Coding Tools - JSON vs. XML

This is going to be a code-heavy chapter for us to go through with a number of interesting tooling and changes to create your Coding Agent!

But first... we have a problem with the JSON-like tool calling that we have used thus far. Imagine that you want to create a tool that runs some code. Creating code typically requires many lines of code, which is not something that is easily supported in JSON. In particular, JSON requires all kinds of special changes in order to make multi-line code possible. Moreover, code often uses these `{` and `}` symbols, which is actually part of the JSON format!

Let's demonstrate this with an example. Imagine we have an `execute_python` function that takes in code as a string which it will run. 

The function that we want to execute is:

```python
def greet(name):
    print("f Hello, {name!}")

greet("World")
```

If the LLM would want to create this code in JSON, it would look a bit like:

```json
'{"tool": "execute_python", "kwargs": {"code": "def greet(name):\n    print(f"Hello, {name}!")\n\ngreet("World")"}}'
```

This is a problem because this string cannot be properly converted to JSON because of the `"` quotes inside `"Hello, {name}!"` and `"World"`. JSON has no idea they're part of the code and thinks they're closing the string. So `json.loads()` sees the string end at `"Hello` and then doesn't know what to do with the rest, giving you a parsing error.

We can check this with the following code, which should give us an error:

In [ ]:
import json

# This is what the LLM actually returns as raw text
response = '{"tool": "execute_python", "kwargs": {"code": "def greet(name):\n    print(f"Hello, {name}!")\n\ngreet("World")"}}'

# This will fail
json.loads(response)

There is the error! 

Now that doesn't mean nothing is possible. We could try to escape all these quotes (`"`) with a backslashes (`\\`) as that would have the JSON properly interpret what does and does not belong to the JSON format. 

Let's check:

In [ ]:
# Fixed: all inner quotes escaped with backslash
response = '{"tool": "execute_python", "kwargs": {"code": "def greet(name):\\n    print(f\\"Hello, {name}!\\")\\n\\ngreet(\\"World\\")"}}'
json.loads(response)

This works!

But, there is a big issue here. The LLM would have to continuously use those backslashes (`\\`), which is error-prone and requires additional tokens. But more importantly, we still have the issue of needing to backslash nearly every special character, like `{}` and `[]` tokens.

So what can we do? We can use a different format for calling tools than JSON that is a bit easier to use for multi-line coding applications, namely XML. XML is a markup language that uses tags like `<tool>` and `</tool>` to wrap content. The nice thing for our use case is that everything between an opening and closing tag is just treated as plain text. As such, newlines, quotes, and indentation will all work. So instead of trying to cram code into the JSON string and hoping that the Agent use those backslashes correctly, we can use `<code>` and `</code>` tags. All code can then be put inside those tags and there is no problem with formatting.

Let's illustrate. Instead of your TinyAgent returning JSON, let's assume it now returns XML:

In [ ]:
import re

# TinyAgent's response
response = """
<tool>execute_python</tool>
<code>
def greet(name):
    print(f"Hello, {name}!")

greet("World")
</code>
"""

# Extract name and code with string manipulation
tool_name = re.search(r"<tool>(.*?)</tool>", response).group(1)
code = re.search(r"<code>(.*?)</code>", response, re.DOTALL).group(1)

# Show
print(f"=== Tool ===\n{tool_name}\n")
print(f"=== Code ==={code}")

Note that in the code above, we use Regular Expression (RegEx) to search for all content with the `<>` and `</>` tags. As you can see, it is much easier now to extract the code here.

> Now you might wonder why we didn't do this in the first place! Large LLM providers typically return JSON tool calls, right?

That can be for a number of reasons. One of them is because model providers like Google, Anthropic, and OpenAI may use something called constrained decoding which limits which tokens the model can output. For instance, if the model tries to output a raw "newline" token, the constrained decoder blocks it (masks the probability to zero) and forces the model to choose the escaped `\n` token instead.

Another reason is that the model may not actually generate JSON, but merely fills it in. So instead of creating the full JSON formatting, it only needs to fill in the following fields:

```json
{
    "tool": "TO_FILL_IN",
    "kwargs": {
        "TO_FILL_IN": "TO_FILL_IN"
    }
}
```

As a result, the `{"tool":...` isn't actually generated by the model because we already know that a tool call should look like that. 

There are more potential reasons, like providers actually using YAML but simply converting it to JSON later on or using extensive libraries to properly parse the JSON outside of the model.

Either way, these are all quite elaborate to implement, so we are going to stick with XML for now and there are a number of things we would need to update:

* `ReACT` - Update instructions on XML tool calling
* `Tools` - Update parsing to use XML
* `Coding Tools` - We need a bunch of tools specific to coding agents, like reading and writing files.

### 2.1 Update `ReACT()`

Let's start with the first component, the instructions given by `ReACT`. Although we keep the same loop of `THOUGHT` -> `ACTION` -> `OBSERVATION`, the `ACTION` will now require **XML** instead of **JSON**. 

The change is rather straightforward, we use the existing `ReACT()` module and only change the `.prompt` property with updated **XML**-based instructions. 

To prevent having to duplicate the `.parse` function, we can have our new `XMLReAct` inherit all functions from `ReAct` by using `XMLReAct(ReAct)`.

Our updated class becomes the following:

In [ ]:
from illustrated_agents.planning import ReAct


class XMLReAct(ReAct):
    """ReAct module."""

    @property
    def prompt(self) -> str:
        return """
# ReACT (Reason and Act)

You are a ReAct agent that performs exactly ONE step per turn.
Make sure to break down a given task into smaller steps and decide whether to use a tool or provide a final answer.

## ReACT Format

You use the following format for each step:

THOUGHT: [Your reasoning about what to do next]
ACTION:
<tool>a_tool_name</tool>
<param_name>value</param_name>

An observation will be provided after each action. You do not generate the observation yourself.

## ReACT Completion

To provide the final answer to the task, use the final_answer tool.
It is the only way to complete the task, else you will be stuck on a loop. So your final output should look like this:

ACTION:
<tool>final_answer</tool>
<answer>insert your final answer here</answer>

Use `final_answer` when you are completely done with all subtasks and have the final answer ready.
You can also use `final_answer` to directly reply to a user's question without using any tools if you think you can answer it directly.
"""

Let's inspect the changes compared to the initial `ReAct` module:

In [ ]:
from illustrated_agents.chapters.ch11 import react_diff; react_diff

Quite straightforward, right? We replaced three instances that relate to JSON-specific instructions and replaced those with XML-specific instructions.

Next up, let's see how we should update `Tools()` to make use of the XML-formatting.

### 2.2 Update `Tools()`

Our previously generated `Tools()` was built around parsing JSON, so now we need to create one that is built around XML instead. We are going to use the same strategy of inheritance as we did with `XMLReAct`. The name changes will be to three functions:

* `.prompt` - Instructions should now relate to XML
* `has_tool_call` - To check whether there is a tool call, it should look for `<tool>` instead of `tool:`
* `parse_tool_call` - We now need to parse XML rather than JSON

Here is the updated code:

In [ ]:
import re
from illustrated_agents.tools import Tools


class XMLTools(Tools):
    """Tool registry for the Agent."""

    @property
    def prompt(self):
        return f"""
# Tools

If needed, you can only use the following tools to assist you in completing tasks:

{self.descriptions}

To use a tool, respond with XML tags:
<tool>tool_name</tool>
<param_name>value</param_name>
"""

    def has_tool_call(self, text: str) -> bool:
        return "<tool>" in text

    def parse_tool_call(self, text: str) -> dict:
        """Parse an XML tool call from text."""
        # Extract the tool name
        tool = re.search(r"<tool>(.*?)</tool>", text, re.DOTALL).group(1).strip()

        # Extract parameters as kwargs
        kwargs = {}
        for match in re.finditer(r"<(\w+)>(.*?)</\1>", text, re.DOTALL):
            name, value = match.group(1), match.group(2).strip()
            if name != "tool":
                kwargs[name] = value

        return {"tool": tool, "kwargs": kwargs}

There are a couple of things happening here that might require a deeper-dive. Let's annotate that important components and what they do:

In [ ]:
from illustrated_agents.chapters.ch11 import tools_annotated; tools_annotated

Let's see how it parses XML in practice by using the example we had before. 

In [ ]:
# LLM-generated response with XML tool call
response = """
<tool>execute_python</tool>
<code>
def greet(name):
    print(f"Hello, {name}!")

greet("World")
</code>
"""

# Parse the XML tool call to extract the tool name and parameters
XMLTools().parse_tool_call(response)

Great! We can parse XML and extract the name and kwargs. This output is the same as is expected by our previous `Tools` object, so there is no need to change how tools are subsequently called.

### 2.2 Create Coding Tools

## 3 - The Styling

In [ ]:
import re
from rich.console import Console
from rich.rule import Rule

console = Console()


class Display:
    """Formats agent events for the terminal."""

    def __init__(self):
        self._status = None

    def __call__(self, event, data=None):

        # Animate thinking
        if event == "thinking":
            self._status = console.status("Thinking...", spinner="dots")
            self._status.start()

        # Print THOUGHT
        elif event == "response":
            self.stop()
            thought = re.search(r"THOUGHT:\s*(.+?)(?=ACTION:|$)", data, re.IGNORECASE | re.DOTALL).group(1).strip()
            console.print(f"  [bold dark_orange]{'THOUGHT':<13}[/][dim italic]{thought}[/]\n")

        # Print ACTION
        elif event == "tool_call" and data:
            tool = data.get("tool")
            if tool != "final_answer":
                args = ", ".join(f"{k}={v!r}" for k, v in data.get("kwargs", {}).items())
                console.print(f"  [bold yellow]{'ACTION':<13}[/][yellow]{tool}({args})[/]")

        # Print OBSERVATION
        elif event == "observation":
            console.print(f"  [bold green]{'OBSERVATION':<13}[/]{data}\n")
            console.print(Rule(style="dim"), end="\n\n")

    def stop(self):
        if self._status:
            self._status.stop()
            self._status = None

In [ ]:
from illustrated_agents.chapters.ch11 import display_annotated; display_annotated

In [ ]:
display = Display()
display("response", "THOUGHT: This is a sample thought.\nACTION: search('example query')")
display("tool_call", {"tool": "search", "kwargs": {"query": "example query"}})
display("observation", "This is a sample observation.")
display.stop()

## 4 - The `TinyAgent`

In [ ]:
from illustrated_agents.chapters.ch11 import tinyagents_diff; tinyagents_diff

## 5 - The Chat Interface

To create an interface, we are going to explore a main component, namely the interface with `rich`.

In [ ]:
def chat(agent):
    """Interactive chat loop — works in both terminal and Jupyter."""
    display = agent.display

    # Agent Overview
    console.print(f"\n  [dim]Tools:[/]  {', '.join(agent.tools.tools.keys())}")
    console.print("  [dim]Type[/] exit [dim]to quit.[/]\n")

    while True:
        # User input
        try:
            query = input("> ").strip()
        except (KeyboardInterrupt, EOFError):
            console.print("  [dim]Goodbye![/]")
            break

        # Skip empty input
        if not query:
            continue

        console.print(f"  [bold]> [/]{query}\n")

        # Exit commands
        if query.lower() in ("exit", "quit"):
            console.print("  [dim]Goodbye![/]")
            break

        # Run agent
        try:
            result = agent.run(query)
            display.stop()
            console.print(f"\n  [bold cyan]{'ANSWER':<13}[/]{result}\n")
        except Exception as e:
            display.stop()
            console.print(f"  [bold red]{'ERROR':<13}[/][red]{e}[/]\n")

## 6 - Putting It All Together

In [ ]:
from illustrated_agents import Memory, Tools, ReAct, Reflector, Skills
from illustrated_agents.chapters.ch11 import TinyAgent

# Tools
def calculator(a: str, b: str) -> float:
    return float(a) + float(b)

def get_weather(location: str) -> str:
    return f"Weather in {location}: Sunny, 72F"

tools = Tools()
tools.add_tool("calculator", calculator, "Adds two numbers: calculator(a, b)")
tools.add_tool("get_weather", get_weather, "Gets weather: get_weather(location)")

# Agent
agent = TinyAgent(
    llm=llm, 
    memory=Memory(), 
    tools=tools,
    planner=ReAct(max_steps=10), 
    reflector=Reflector(interval=20),
    skills=Skills(), 
    display=Display(),
)

In [ ]:
# Start interactive chat
chat(agent)

## 7 - The "Real" CLI

TODO: Add description of how the `cli.py` works.